In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [2]:
df = pd.read_csv('data/ai4i2020.csv')

In [3]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
df.describe()

,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,300.004930,310.005560,1538.776100,39.986910,107.951000,0.033900,0.004600,0.011500,0.009500,0.009800,0.00190
std,2886.89568,2.000259,1.483734,179.284096,9.968934,63.654147,0.180981,0.067671,0.106625,0.097009,0.098514,0.04355
min,1.00000,295.300000,305.700000,1168.000000,3.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,2500.75000,298.300000,308.800000,1423.000000,33.200000,53.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,5000.50000,300.100000,310.100000,1503.000000,40.100000,108.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,7500.25000,301.500000,311.100000,1612.000000,46.800000,162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,10000.00000,304.500000,313.800000,2886.000000,76.600000,253.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


In [5]:
df.isnull().sum()

UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

In [6]:
df.shape

(10000, 14)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
me

In [8]:
print(df['Machine failure'].value_counts())

print(df['Machine failure'].value_counts(normalize=True) * 100)

Machine failure
0    9661
1     339
Name: count, dtype: int64
Machine failure
0    96.61
1     3.39
Name: proportion, dtype: float64


In [9]:
# Visualization of plots 
DATA_PATH  = "data/ai4i2020.csv"
OUTPUT_DIR = "outputs"
LABEL_COL  = "machine_failure"

os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")
COLORS = {"normal": "steelblue", "anomaly": "tomato"}

In [10]:
df = pd.read_csv(DATA_PATH)
df.columns = [
    c.strip().lower()
     .replace(" ", "_")
     .replace("[", "")
     .replace("]", "")
    for c in df.columns
]

# Drop identifier columns, not useful for EDA or modelling
drop_cols = ["udi", "product_id"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

In [11]:
df.columns

Index(['type', 'air_temperature_k', 'process_temperature_k',
       'rotational_speed_rpm', 'torque_nm', 'tool_wear_min', 'machine_failure',
       'twf', 'hdf', 'pwf', 'osf', 'rnf'],
      dtype='str')

In [12]:
# Renaming sensor columns to clean names , strips suffixes like _k
rename_map = {}
for col in df.columns:
    if "air_temperature"     in col: rename_map[col] = "air_temp"
    if "process_temperature" in col: rename_map[col] = "process_temp"
    if "rotational_speed"    in col: rename_map[col] = "rotational_speed"
    if "torque"              in col: rename_map[col] = "torque"
    if "tool_wear"           in col: rename_map[col] = "tool_wear"
df = df.rename(columns=rename_map)

In [13]:
# Encode type for numeric plots
type_col = "type" if "type" in df.columns else None
if type_col:
    df["type_encoded"] = df[type_col].map({"L": 0, "M": 1, "H": 2})

SENSOR_FEATURES = [
    "air_temp", "process_temp", "rotational_speed", "torque", "tool_wear"
]

df["temp_difference"] = df["process_temp"] - df["air_temp"]
df["power_watts"]     = df["torque"] * (df["rotational_speed"] * 2 * np.pi / 60)
df["tool_torque"]     = df["tool_wear"] * df["torque"]
df["power_deviation"] = np.abs(df["power_watts"] - 6250)

ENG_FEATURES = ["temp_difference", "power_watts", "tool_torque", "power_deviation"]

# All numeric features for KDE grid (sensors + engineered)
ALL_FEATURES = SENSOR_FEATURES + ENG_FEATURES

In [14]:
print(" ///Dataset overview")
print(f"Shape: {df.shape}")
print(f"\nStatistics:\n{df[ALL_FEATURES].describe().round(2)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nClass distribution:\n{df[LABEL_COL].value_counts()}")
print(f"\nClass ratio (anomaly %): {df[LABEL_COL].mean()*100:.2f}%")

if type_col:
    print(f"\nProduct type distribution:\n{df[type_col].value_counts()}")

 ///Dataset overview
Shape: (10000, 17)

Statistics:
       air_temp  process_temp  rotational_speed    torque  tool_wear  \
count   10000.0      10000.00          10000.00  10000.00   10000.00   
mean      300.0        310.01           1538.78     39.99     107.95   
std         2.0          1.48            179.28      9.97      63.65   
min       295.3        305.70           1168.00      3.80       0.00   
25%       298.3        308.80           1423.00     33.20      53.00   
50%       300.1        310.10           1503.00     40.10     108.00   
75%       301.5        311.10           1612.00     46.80     162.00   
max       304.5        313.80           2886.00     76.60     253.00   

       temp_difference  power_watts  tool_torque  power_deviation  
count          10000.0     10000.00     10000.00         10000.00  
mean              10.0      6279.74      4314.66           849.29  
std                1.0      1067.42      2826.57           647.22  
min                7.6    

In [15]:
# KDE feature distributions by class 
# 3 rows × 3 cols for 9 features (5 sensor and 4 engineered)
n_cols = 3
n_rows = int(np.ceil(len(ALL_FEATURES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3.5))
axes = axes.flatten()

for ax, col in zip(axes, ALL_FEATURES):
    for label, color, name in [
        (0, COLORS["normal"],  "Normal"),
        (1, COLORS["anomaly"], "Failure")
    ]:
        sns.kdeplot(
            data=df[df[LABEL_COL] == label],
            x=col,
            ax=ax,
            fill=True,
            alpha=0.4,
            color=color,
            label=name
        )
    ax.set_title(col, fontweight="bold")
    ax.set_xlabel("")
    ax.legend(fontsize=8)

for ax in axes[len(ALL_FEATURES):]:
    ax.set_visible(False)

plt.suptitle("Feature distributions: Normal vs Failure", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved plot")

Saved plot


In [16]:
# Correlation with target failure (bar chart) 
corr = df[ALL_FEATURES + [LABEL_COL]].corr()[LABEL_COL].drop(LABEL_COL).sort_values()

fig, ax = plt.subplots(figsize=(7, 5))
colors_bar = [COLORS["anomaly"] if v > 0 else COLORS["normal"] for v in corr.values]
corr.plot(kind="barh", ax=ax, color=colors_bar, edgecolor="white")

# Annotate each bar with its value
for i, (val, name) in enumerate(zip(corr.values, corr.index)):
    offset = 0.003 if val >= 0 else -0.003
    ha     = "left" if val >= 0 else "right"
    ax.text(val + offset, i, f"{val:.3f}", va="center", ha=ha, fontsize=9)

ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Correlation with machine_failure", fontweight="bold")
ax.set_xlabel("Pearson correlation")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_target_correlations.png", dpi=150)
plt.close()
print("Saved plot")

Saved plot
